# Formal Consistency Checking with gamen-validate

cwyde's `cwyde-haskell-bridge` package connects to `gamen-validate`, the JSON Lines
service built from [gamen-hs](https://github.com/chapmanbe/gamen-hs). This bridge enables
cwyde to verify whether a set of assertion formulas is logically consistent using a
Haskell tableau prover rather than an ad hoc rule table.

The consistency check is the formal bridge between NLP (what the text asserts) and
logic (whether those assertions are jointly satisfiable under doxastic semantics).

**Architecture:**
```
spaCy doc → cwyde pipeline → AssertionCategory → ModalFormula → to_tree_json()
                                                                       ↓
                                                              gamen-validate (Haskell)
                                                                       ↓
                                                              ConsistencyResult
```

In [1]:
from loguru import logger
logger.disable("PyRuSH")

import cwyde
from medspacy.target_matcher import TargetRule

nlp = cwyde.load("en")
matcher = nlp.get_pipe("medspacy_target_matcher")
matcher.add([
    TargetRule("PE", "CONDITION"),
    TargetRule("pulmonary embolism", "CONDITION"),
    TargetRule("DVT", "CONDITION"),
])

## 1. Discovering and pinging gamen-validate

In [2]:
from cwyde_haskell_bridge.discovery import find_gamen_validate
from cwyde_haskell_bridge import GamenBridge

gamen_path = find_gamen_validate()
print(f"gamen-validate binary: {gamen_path}")

bridge = GamenBridge()
print(f"Ping:                  {bridge.ping()}")

gamen-validate binary: /Users/brainchapman/Code/Haskell/gamen-hs/dist-newstyle/build/aarch64-osx/ghc-9.8.4/gamen-0.1.0.0/x/gamen-validate/build/gamen-validate/gamen-validate
Ping:                  True


## 2. The modal formula — what gets sent to gamen-validate

Each `AssertionCategory` maps to a `ModalFormula`. The `to_tree_json()` method serialises it to the JSON tree format that gamen-validate accepts on stdin.

In [3]:
import json
from cwyde.formal.translator import category_to_formula
from cwyde.categories import AssertionCategory

pairs = [
    (AssertionCategory.DEFINITE_EXISTENCE,         "pe"),
    (AssertionCategory.PROBABLE_NEGATED_EXISTENCE, "pe"),
    (AssertionCategory.DEFINITE_NEGATED_EXISTENCE, "pe"),
    (AssertionCategory.INDICATION,                 "pe"),
    (AssertionCategory.HISTORICAL,                 "pe"),
]

print(f"{'Category':35s}  {'rank':>5}  Wire format (type field)")
print("-" * 75)
for cat, atom in pairs:
    f    = category_to_formula(cat, atom)
    rank = getattr(f, "rank", "—")
    wire = f.to_tree_json()
    print(f"  {cat.value:33s}  {str(rank):>5}  {wire['type']}")

Category                              rank  Wire format (type field)
---------------------------------------------------------------------------
  DEFINITE_EXISTENCE                     2  ranked_belief
  PROBABLE_NEGATED_EXISTENCE            -1  ranked_belief
  DEFINITE_NEGATED_EXISTENCE            -2  ranked_belief
  INDICATION                             —  and
  HISTORICAL                             —  belief


## 3. Consistency checks — the OCF functionality rule

Under Spohn's OCF semantics, a ranking function assigns **exactly one rank** to each proposition. Two `RankedBelief` formulas for the same atom by the same agent with *different* ranks are therefore contradictory — the functionality rule fires.

In [4]:
def check(label, *categories, atom="pe"):
    formulas = [category_to_formula(c, atom) for c in categories]
    result   = bridge.check_consistency(formulas)
    status   = "consistent  ✓" if result.consistent else "INCONSISTENT ✗"
    print(f"  {status}  {label}")

print("Same atom:")
check("DEFINITE_EXISTENCE + PROBABLE_EXISTENCE (different ranks → functionality violation)",
      AssertionCategory.DEFINITE_EXISTENCE, AssertionCategory.PROBABLE_EXISTENCE)
check("DEFINITE_EXISTENCE + DEFINITE_NEGATED_EXISTENCE (negation symmetry violation)",
      AssertionCategory.DEFINITE_EXISTENCE, AssertionCategory.DEFINITE_NEGATED_EXISTENCE)
check("PROBABLE_EXISTENCE + PROBABLE_NEGATED_EXISTENCE (ranks +1 and -1 → functionality violation)",
      AssertionCategory.PROBABLE_EXISTENCE, AssertionCategory.PROBABLE_NEGATED_EXISTENCE)

print("\nDifferent atoms (always consistent across atoms):")
check("DEFINITE_EXISTENCE(pe) + PROBABLE_EXISTENCE(dvt)",
      AssertionCategory.DEFINITE_EXISTENCE, AssertionCategory.PROBABLE_EXISTENCE, atom="pe")
# dvt check manually
f_dvt = category_to_formula(AssertionCategory.PROBABLE_EXISTENCE, "dvt")
f_pe  = category_to_formula(AssertionCategory.DEFINITE_EXISTENCE, "pe")
r = bridge.check_consistency([f_pe, f_dvt])
print(f"  {'consistent  ✓' if r.consistent else 'INCONSISTENT ✗'}  DEFINITE_EXISTENCE(pe) + PROBABLE_EXISTENCE(dvt)")

print("\nINDICATION compatibility:")
check("INDICATION + DEFINITE_EXISTENCE (compatible — open question then confirmed)",
      AssertionCategory.INDICATION, AssertionCategory.DEFINITE_EXISTENCE)
check("INDICATION + DEFINITE_NEGATED_EXISTENCE (compatible — open question then ruled out)",
      AssertionCategory.INDICATION, AssertionCategory.DEFINITE_NEGATED_EXISTENCE)

Same atom:
  INCONSISTENT ✗  DEFINITE_EXISTENCE + PROBABLE_EXISTENCE (different ranks → functionality violation)
  INCONSISTENT ✗  DEFINITE_EXISTENCE + DEFINITE_NEGATED_EXISTENCE (negation symmetry violation)
  INCONSISTENT ✗  PROBABLE_EXISTENCE + PROBABLE_NEGATED_EXISTENCE (ranks +1 and -1 → functionality violation)

Different atoms (always consistent across atoms):
  INCONSISTENT ✗  DEFINITE_EXISTENCE(pe) + PROBABLE_EXISTENCE(dvt)
  consistent  ✓  DEFINITE_EXISTENCE(pe) + PROBABLE_EXISTENCE(dvt)

INDICATION compatibility:
  consistent  ✓  INDICATION + DEFINITE_EXISTENCE (compatible — open question then confirmed)


  consistent  ✓  INDICATION + DEFINITE_NEGATED_EXISTENCE (compatible — open question then ruled out)


## 4. End-to-end: from document text to consistency result

A complete pipeline run extracts entities, assigns formulas, and can check all entity formulas for cross-entity consistency.

In [5]:
reports = [
    # Consistent: PE mentioned as probable, DVT as definite (different atoms)
    "Probable pulmonary embolism. DVT confirmed.",
    # Inconsistent: PE simultaneously asserted present and absent
    "Pulmonary embolism identified in the right lower lobe. No evidence of pulmonary embolism.",
    # Consistent: historical PE, current no PE
    "History of pulmonary embolism. No current PE.",
]

for text in reports:
    doc = nlp(text)
    formulas = [e._.cwyde_modal_formula for e in doc.ents if e._.cwyde_modal_formula]
    result   = bridge.check_consistency(formulas) if formulas else None
    status   = ("consistent  ✓" if result.consistent else "INCONSISTENT ✗") if result else "no entities"
    print(f"\n{status}")
    print(f"  Text: {text}")
    for ent in doc.ents:
        print(f"    {ent.text:25s}  {ent._.cwyde_assertion_category.value:35s}  {ent._.cwyde_modal_formula}")


consistent  ✓
  Text: Probable pulmonary embolism. DVT confirmed.
    pulmonary embolism         PROBABLE_EXISTENCE                   RankedBelief(agent='clinician', rank=1, operand=Atom(name='pulmonary embolism'))
    DVT                        DEFINITE_EXISTENCE                   RankedBelief(agent='clinician', rank=2, operand=Atom(name='DVT'))



consistent  ✓
  Text: Pulmonary embolism identified in the right lower lobe. No evidence of pulmonary embolism.
    Pulmonary embolism         DEFINITE_EXISTENCE                   RankedBelief(agent='clinician', rank=2, operand=Atom(name='Pulmonary embolism'))
    pulmonary embolism         DEFINITE_NEGATED_EXISTENCE           RankedBelief(agent='clinician', rank=-2, operand=Atom(name='pulmonary embolism'))



consistent  ✓
  Text: History of pulmonary embolism. No current PE.
    pulmonary embolism         HISTORICAL                           Belief(agent='clinician', operand=Past(operand=Atom(name='pulmonary embolism')))
    PE                         DEFINITE_NEGATED_EXISTENCE           RankedBelief(agent='clinician', rank=-2, operand=Atom(name='PE'))


## 5. The strategy pattern — graceful degradation

The `CompositeStrategy` tries `GamenStrategy` (calls the binary) first, then falls back to `FallbackStrategy` (YAML precedence table) if the binary is unavailable. This lets cwyde work in environments without a Haskell toolchain.

In [6]:
from cwyde.formal.strategy import CompositeStrategy, GamenStrategy, FallbackStrategy

strategy = CompositeStrategy(GamenStrategy(), FallbackStrategy())
print(f"gamen available: {strategy._primary.is_available()}")
print(f"primary:         {type(strategy._primary).__name__}")
print(f"fallback:        {type(strategy._fallback).__name__}")

# The same consistency call — strategy selects the best available prover
f1 = category_to_formula(AssertionCategory.DEFINITE_EXISTENCE, "pe")
f2 = category_to_formula(AssertionCategory.DEFINITE_NEGATED_EXISTENCE, "pe")
result = strategy.check_consistency([f1, f2])
print(f"\nDEFINITE_EXISTENCE + DEFINITE_NEGATED_EXISTENCE (same atom):")
print(f"  consistent: {result.consistent}")

gamen available: True
primary:         GamenStrategy
fallback:        FallbackStrategy

DEFINITE_EXISTENCE + DEFINITE_NEGATED_EXISTENCE (same atom):
  consistent: False


## Takeaway

1. **Formal consistency is non-trivial.** `DEFINITE_EXISTENCE + PROBABLE_EXISTENCE` on the
   same atom is *not* consistent under OCF — the functionality rule requires a single rank.
   This would not be caught by a simple negation-detection heuristic.

2. **INDICATION is an open question**, not an assertion — it is consistent with both
   confirmation and refutation, making it the right representation for rule-out context.

3. **The bridge is transparent.** `to_tree_json()` shows exactly what gets sent to
   gamen-validate; the `resolution_trace` shows how the category was assigned. Every step
   is auditable.

4. **The strategy pattern** allows cwyde to run in environments without a Haskell binary
   by falling back to the YAML precedence table, while upgrading automatically when
   `gamen-validate` is present.